In [ ]:
# Part A — Creating the Date Spine
# Q1. Load the store dataset
import pandas as pd
import numpy as np

STORE_PATH = "store_transactions_sparse.csv"
ENERGY_PATH = "energy_usage_hourly.csv"
store = pd.read_csv(STORE_PATH,parse_dates=["date"])

print("Rows:", len(store))
print("Minimum date:", store["date"].min())
print("Maximum date:", store["date"].max())
print("="*50)
calendar_days = (store["date"].max() - store["date"].min()).days + 1
missing_days = calendar_days - len(store)
print("Calendar days:", calendar_days)
print("Difference:", missing_days)

# Q2. Build a complete daily date spine
spine = pd.date_range(start=store["date"].min(),end=store["date"].max(),freq="D")
store_spined = (store.set_index("date").reindex(spine))
store_spined.index.name = "date"

print(store_spined.shape)
print("="*50)
print(store_spined.head(30))
print("="*30)
print(store_spined.isnull().sum())

# Q3. Missing store days — 0 or interpolation?
store_spined["transactions"] = (store_spined["transactions"].fillna(0))
store_spined["revenue"] = (store_spined["revenue"].fillna(0))

# Store closed
#       ↓
# No transactions
#       ↓
# transactions = 0
# revenue = 0

# Q4. Add was_closed

store_spined = (store.set_index("date").reindex(spine))
store_spined.index.name = "date"
store_spined["was_closed"] = (store_spined["revenue"].isna())
store_spined["transactions"] = (store_spined["transactions"].fillna(0))
store_spined["revenue"] = (store_spined["revenue"].fillna(0))
print(store_spined["was_closed"].value_counts())
print(store_spined[store_spined["was_closed"] == True])

# Q5. Energy hourly spine
energy = pd.read_csv(ENERGY_PATH,parse_dates=["timestamp"])
energy = energy.sort_values("timestamp")
hourly_spine = pd.date_range(start=energy["timestamp"].min(),end=energy["timestamp"].max(),freq="h")
energy_spined = (energy.set_index("timestamp").reindex(hourly_spine))
energy_spined.index.name = "timestamp"

print("Expected hours:", len(hourly_spine))
print("Actual rows:", len(energy))
print("Missing hours:", energy_spined["kwh"].isna().sum())

energy_spined["kwh"] = (energy_spined["kwh"].interpolate(method="time"))

# Part B — Lag and Lead Functions
# Q6. lag_1 and lag_7
store_spined["lag_1"] = (store_spined["revenue"].shift(1))
store_spined["lag_7"] = (store_spined["revenue"].shift(7))
store_spined.head(9)

# Q7. lead_1
store_spined["lead_1"] = (store_spined["revenue"].shift(-1))

# Q8. Day-over-day change
store_spined["dod_change"] = (store_spined["revenue"]- store_spined["lag_1"])
print(store_spined["dod_change"].head())

print("="*40)
# percentage change
store_spined["dod_pct_change"] = (store_spined["revenue"].pct_change() * 100)
print(store_spined["dod_pct_change"].head())

# Q9. 365-day lag ወይስ 7-day lag?


# Q10. Record Day
store_spined["lag_14"] = (
    store_spined["revenue"].shift(14)
)

store_spined["record_day"] = (
    (store_spined["revenue"] > store_spined["lag_7"]) &
    (store_spined["revenue"] > store_spined["lag_14"])
)

# Part C — Rolling Windows
# Q11. 7-day trailing rolling mean

store_spined["rolling_7_mean"] = (
    store_spined["revenue"]
    .rolling(window=7)
    .mean()
)
print(store_spined["rolling_7_mean"].tail())

# Q12. Centered rolling mean
store_spined["rolling_7_centered"] = (
    store_spined["revenue"]
    .rolling(window=7, center=True)
    .mean()
)
print(store_spined["rolling_7_centered"])

# Q13. Rolling standard deviation + anomaly flag
store_spined["rolling_7_std"] = (
    store_spined["revenue"]
    .rolling(window=7)
    .std()
)

store_spined["unusual"] = (
    abs(
        store_spined["revenue"]- store_spined["rolling_7_mean"])>2 * store_spined["rolling_7_std"]
)

print(store_spined["unusual"])

# Q14. min_periods=1
# rolling(7)

# store_spined["rolling_7_mean_min1"] = (
#     store_spined["revenue"]
#     .rolling(window=7, min_periods=1)
#     .mean()
# )

# Q15. Expanding mean
store_spined["expanding_mean"] = (
    store_spined["revenue"]
    .expanding()
    .mean()
)

# Part D — Aggregating to Different Frequencies
# Q16. Daily → Weekly

weekly = (
    store_spined
    .resample("W")
    .agg(
        transactions=("transactions", "sum"),
        revenue=("revenue", "sum")
    )
)

print(weekly)

# Q17. Average Transaction Value
monthly = (
    store_spined
    .resample("MS")
    .agg(
        revenue=("revenue", "sum"),
        transactions=("transactions", "sum")
    )
)

monthly["average_transaction_value"] = (
    monthly["revenue"]
    / monthly["transactions"]
)

# Q18. Energy Hourly → Daily → Weekly
# Daily Total
energy_daily = (
    energy_spined["kwh"]
    .resample("D")
    .sum()
)

# Weekly Total
energy_weekly = (
    energy_spined["kwh"]
    .resample("W")
    .sum()
)

# Daily → Hourly
hourly_from_daily = (
    energy_daily
    .resample("h")
    .asfreq()
)
hourly_from_daily.interpolate()

# Q19. Average Revenue by Weekday
weekday_avg = (
    store_spined
    .groupby(store_spined.index.dayofweek)["revenue"]
    .mean()
)

weekday_avg.index = [
    "Monday",
    "Tuesday",
    "Wednesday",
    "Thursday",
    "Friday",
    "Saturday",
    "Sunday"
]

weekday_avg = weekday_avg.reset_index()
weekday_avg.columns = [
    "weekday",
    "average_revenue"
]

print(weekday_avg)

# Q20. Monthly Revenue — Two Methods
# Method A — Direct
monthly_direct = (
    store_spined["revenue"]
    .resample("MS")
    .sum()
)
# Method B — Weekly first
weekly_revenue = (
    store_spined["revenue"]
    .resample("W")
    .sum()
)

monthly_from_weekly = (
    weekly_revenue
    .resample("MS")
    .sum()
)

# Part E — Q21 Mini Integration Challenge
import pandas as pd
import numpy as np

# ============================================================
# 1. LOAD DATA
# ============================================================

STORE_PATH = "store_transactions_sparse.csv"

store = pd.read_csv(
    STORE_PATH,
    parse_dates=["date"]
)

store = store.sort_values("date")


# ============================================================
# 2. CREATE COMPLETE DAILY DATE SPINE
# ============================================================

date_spine = pd.date_range(
    start=store["date"].min(),
    end=store["date"].max(),
    freq="D"
)

store_clean = (
    store
    .set_index("date")
    .reindex(date_spine)
)

store_clean.index.name = "date"


# ============================================================
# 3. IDENTIFY CLOSED / MISSING DAYS
# ============================================================

store_clean["was_closed"] = (
    store_clean["revenue"].isna()
)


# ============================================================
# 4. FILL CLOSED DAYS WITH ZERO
# ============================================================

store_clean["transactions"] = (
    store_clean["transactions"].fillna(0)
)

store_clean["revenue"] = (
    store_clean["revenue"].fillna(0)
)


# ============================================================
# 5. 7-DAY TRAILING ROLLING AVERAGE
# ============================================================

store_clean["rolling_7d_avg_revenue"] = (
    store_clean["revenue"]
    .rolling(window=7)
    .mean()
)


# ============================================================
# 6. RESAMPLE DAILY DATA TO WEEKLY
# ============================================================

weekly = (
    store_clean
    .resample("W")
    .agg(
        total_revenue=("revenue", "sum"),
        total_transactions=("transactions", "sum"),
        average_daily_revenue=("revenue", "mean")
    )
)


# ============================================================
# 7. 4-WEEK ROLLING AVERAGE OF WEEKLY REVENUE
# ============================================================

weekly["rolling_4w_avg_revenue"] = (
    weekly["total_revenue"]
    .rolling(window=4)
    .mean()
)


# ============================================================
# 8. WEEK-OVER-WEEK PERCENT CHANGE
# ============================================================

weekly["wow_pct_change"] = (
    weekly["total_revenue"]
    .pct_change()
    * 100
)


# ============================================================
# 9. RESET INDEX
# ============================================================

weekly = weekly.reset_index()


# ============================================================
# 10. DISPLAY FINAL TABLE
# ============================================================

print(weekly.to_string(index=False))

